In [0]:
%pip install nemosis

   
# Bronze Layer Ingestion

Ingesting 5 AEMO tables from 2021 (5MS start) to present:

## Regional Dispatch Tables (Cell 3)
* **DISPATCHPRICE** → `nsw_dispatch_5min` - 5-minute price data for NSW
* **DISPATCHREGIONSUM** → `nsw_demand` - Regional demand and generation totals

## Reference Tables (Cells 4-5)
* **DUDETAILSUMMARY** → `nsw_dictionary` - Dispatch unit metadata (all regions)
  * Uses dictionary records approach to avoid PyArrow ChunkedArray errors
* **DUDETAIL** → `nsw_dictionary_second` - Unit capacity and operational config (versioned)
  * Time-versioned reference table with EFFECTIVEDATE + VERSIONNO
  * Contains REGISTEREDCAPACITY, MAXCAPACITY, AGCCAPABILITY

## SCADA Data (Cell 6)
* **DISPATCH_UNIT_SCADA** → `nsw_scada` - 5-minute SCADA telemetry per unit
  * **Monthly batch processing** to prevent memory exhaustion
  * Processes ~1 month at a time instead of full years

In [0]:
import os
from nemosis import dynamic_data_compiler
import pandas as pd
from datetime import datetime
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.sql.types import DoubleType, IntegerType

# --- 1. ENVIRONMENT & AUTHENTICATION ---
# This block detects if you are in VS Code or the Databricks Web UI
# If you are in the Databricks Web UI, it should already has spark and dbutils
try:
    # Check if spark and dbutils exist AND are functional
    spark.sql("SELECT 1").collect()
    dbutils
    print("Environment: Databricks Notebook")
except:
    # If they don't exist or are broken, recreate for the current environment
    try:
        # Try to get the default Databricks Notebook spark session
        spark = SparkSession.builder.getOrCreate()
        from pyspark.dbutils import DBUtils
        dbutils = DBUtils(spark)
        print("Environment: Databricks Notebook (recreated session)")
    except:
        # Fall back to Databricks Connect for VS Code
        from databricks.connect import DatabricksSession
        from databricks.sdk import WorkspaceClient
        
        spark = DatabricksSession.builder.getOrCreate()
        dbutils = WorkspaceClient().dbutils
        print("Environment: VS Code")

# --- 2. PARAMETER INITIALIZATION ---
# Note: When this notebook runs as part of ingest_bronze_job,
# the base_parameters from databricks.yml will override these defaults
dbutils.widgets.text("catalog", "energy_endava_193")
dbutils.widgets.text("schema", "default") 

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")

print(f"Using catalog: {catalog}")
print(f"Using schema: {schema}")

# Set the Spark context to your specific sandbox
spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema}")
spark.sql(f"USE SCHEMA {schema}")

print(f"✅ Ingestion Target: {catalog}.{schema}")

# --- 3. INGESTION CONFIGURATION ---
# DISPATCHPRICE and DISPATCHREGIONSUM only (DUDETAILSUMMARY ingested separately)
TABLE_TO_INGEST = {
    "DISPATCHPRICE": "nsw_dispatch_5min",
    "DISPATCHREGIONSUM": "nsw_demand"
}
CACHE_DIR = "/tmp/nemosis_cache"
REGION = ["NSW1", "QLD1", "TAS1", "SA1", "VIC1"]

if not os.path.exists(CACHE_DIR):
    os.makedirs(CACHE_DIR)

# 5-Minute Settlement (5MS) officially started Oct 1, 2021
current_year = datetime.now().year # 2026
years_to_process = range(2021, current_year + 1)

# Drop existing tables to reingest the data
for TABLE_NAME, TARGET_TABLE in TABLE_TO_INGEST.items():
    spark.sql(f"DROP TABLE IF EXISTS {TARGET_TABLE}")
    print(f"🗑️ Dropped {TARGET_TABLE} for fresh ingestion")

# --- 4. DATA PROCESSING LOOP ---
for year in years_to_process:
    for region in REGION:
        # Adjust start date for 2021 to capture the 5MS transition
        start_date = f"{year}/10/01 00:00:00" if year == 2021 else f"{year}/01/01 00:00:00"
        
        # Handle the "till now" requirement for 2026
        if year == current_year:
            end_date = datetime.now().strftime("%Y/%m/%d %H:%M:%S")
        else:
            end_date = f"{year}/12/31 23:55:00"

        for TABLE_NAME, TARGET_TABLE in TABLE_TO_INGEST.items():
        
            print(f"--- ⏳ Downloading {TABLE_NAME}: {start_date} to {end_date} ---")
            
            try:
                # Download from AEMO
                df_raw = dynamic_data_compiler(
                    start_time=start_date,
                    end_time=end_date,
                    table_name=TABLE_NAME,
                    raw_data_location=CACHE_DIR
                )
                
                # Filter for NSW region only
                if 'REGIONID' in df_raw.columns:
                    df_filtered = df_raw[df_raw['REGIONID'] == region]
                    print(f"ℹ️  Filtered to {region}: {len(df_filtered):,} rows")
                else:
                    df_filtered = df_raw
                    print(f"⚠️  No REGIONID column - keeping all data")
                
                # Create Spark DataFrame
                spark_df = spark.createDataFrame(df_filtered)
                
                # Define columns that should ALWAYS be Double (all numeric measures)
                # Exclude only true identifier/counter columns that should remain as integers
                INTEGER_COLUMNS = {'DISPATCHINTERVAL', 'INTERVENTION'}
                
                # Cast all numeric columns (except integer identifiers) to DoubleType for consistency
                for field in spark_df.schema.fields:
                    if field.name not in INTEGER_COLUMNS:
                        # Check if the field is a numeric type (int or float)
                        from pyspark.sql.types import IntegralType, FractionalType
                        if isinstance(field.dataType, (IntegralType, FractionalType)):
                            spark_df = spark_df.withColumn(field.name, col(field.name).cast(DoubleType()))
                
                # Write to Delta Lake
                # 'append' mode with 'mergeSchema' ensures we don't crash if AEMO adds columns
                if year == 2021:
                    spark_df.write.format("delta").mode("overwrite").saveAsTable(TARGET_TABLE)
                else:
                    spark_df.write.format("delta").mode("append") \
                        .option("mergeSchema", "true") \
                        .saveAsTable(TARGET_TABLE)
                        
                print(f"✅ Successfully saved {year} data to {TARGET_TABLE}")
            
            except Exception as e:
                print(f"❌ Error in {year} for {TABLE_NAME}: {e}")

print(f"\n🚀 DISPATCHPRICE and DISPATCHREGIONSUM ingestion complete at {catalog}.{schema}")

In [0]:
import os
from nemosis import dynamic_data_compiler
import pandas as pd
from datetime import datetime
from pyspark.sql.functions import col
from pyspark.sql.types import DoubleType

# --- 1. ENVIRONMENT & AUTHENTICATION ---
try:
    spark.sql("SELECT 1").collect()
    dbutils
    print("Environment: Databricks Notebook")
except:
    try:
        from pyspark.sql import SparkSession
        from pyspark.dbutils import DBUtils
        spark = SparkSession.builder.getOrCreate()
        dbutils = DBUtils(spark)
        print("Environment: Databricks Notebook (recreated session)")
    except:
        from databricks.connect import DatabricksSession
        from databricks.sdk import WorkspaceClient
        spark = DatabricksSession.builder.getOrCreate()
        dbutils = WorkspaceClient().dbutils
        print("Environment: VS Code")

# --- 2. PARAMETER INITIALIZATION ---
dbutils.widgets.text("catalog", "energy_endava_193")
dbutils.widgets.text("schema", "default") 

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")

print(f"Using catalog: {catalog}")
print(f"Using schema: {schema}")

spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema}")
spark.sql(f"USE SCHEMA {schema}")

print(f"✅ Ingestion Target: {catalog}.{schema}")

# --- 3. DUDETAILSUMMARY CONFIGURATION ---
TABLE_NAME = "DUDETAILSUMMARY"
TARGET_TABLE = "nsw_dictionary"
CACHE_DIR = "/tmp/nemosis_cache"

if not os.path.exists(CACHE_DIR):
    os.makedirs(CACHE_DIR)

# 5-Minute Settlement (5MS) officially started Oct 1, 2021
current_year = datetime.now().year
years_to_process = range(2021, current_year + 1)

# Drop existing table to reingest fresh data
spark.sql(f"DROP TABLE IF EXISTS {TARGET_TABLE}")
print(f"🗑️ Dropped {TARGET_TABLE} for fresh ingestion")

# --- 4. DATA PROCESSING LOOP ---
for year in years_to_process:
    # Adjust start date for 2021 to capture the 5MS transition
    start_date = f"{year}/10/01 00:00:00" if year == 2021 else f"{year}/01/01 00:00:00"
    
    # Handle the "till now" requirement for current year
    if year == current_year:
        end_date = datetime.now().strftime("%Y/%m/%d %H:%M:%S")
    else:
        end_date = f"{year}/12/31 23:55:00"

    print(f"--- ⏳ Downloading {TABLE_NAME}: {start_date} to {end_date} ---")
    
    try:
        # Download from AEMO
        df_raw = dynamic_data_compiler(
            start_time=start_date,
            end_time=end_date,
            table_name=TABLE_NAME,
            raw_data_location=CACHE_DIR
        )
        
        # DUDETAILSUMMARY is a reference/lookup table
        # Keep ALL regions - needed for joins with regional dispatch data
        print(f"ℹ️  Reference table - keeping all regions: {len(df_raw):,} rows")
        
        # ⚠️ PYARROW FIX: Dictionary Records Approach
        # Convert to dict records and recreate DataFrame to avoid PyArrow ChunkedArray issues
        # This approach was verified successful in DEBUG cell
        print(f"ℹ️  Applying PyArrow fix: converting to dict records...")
        records = df_raw.to_dict('records')
        df_clean = pd.DataFrame(records)
        df_clean = df_clean.reset_index(drop=True)
        print(f"ℹ️  Recreated DataFrame from {len(records):,} records")
        
        # Create Spark DataFrame
        spark_df = spark.createDataFrame(df_clean)
        
        # Cast numeric columns to DoubleType for consistency
        # DUDETAILSUMMARY numeric fields: TRANSMISSIONLOSSFACTOR, DISTRIBUTIONLOSSFACTOR, 
        # MAX_RAMP_RATE_UP, MAX_RAMP_RATE_DOWN
        from pyspark.sql.types import IntegralType, FractionalType
        for field in spark_df.schema.fields:
            if isinstance(field.dataType, (IntegralType, FractionalType)):
                spark_df = spark_df.withColumn(field.name, col(field.name).cast(DoubleType()))
        
        # Write to Delta Lake
        if year == 2021:
            spark_df.write.format("delta").mode("overwrite").saveAsTable(TARGET_TABLE)
        else:
            spark_df.write.format("delta").mode("append") \
                .option("mergeSchema", "true") \
                .saveAsTable(TARGET_TABLE)
                
        print(f"✅ Successfully saved {year} data to {TARGET_TABLE}")
    
    except Exception as e:
        print(f"❌ Error in {year} for {TABLE_NAME}: {e}")
        import traceback
        traceback.print_exc()

print(f"\n🚀 {TABLE_NAME} ingestion complete at {catalog}.{schema}.{TARGET_TABLE}")

In [0]:
import os
from nemosis import dynamic_data_compiler
import pandas as pd
from datetime import datetime
from pyspark.sql.functions import col
from pyspark.sql.types import DoubleType

# --- 1. ENVIRONMENT & AUTHENTICATION ---
try:
    spark.sql("SELECT 1").collect()
    dbutils
    print("Environment: Databricks Notebook")
except:
    try:
        from pyspark.sql import SparkSession
        from pyspark.dbutils import DBUtils
        spark = SparkSession.builder.getOrCreate()
        dbutils = DBUtils(spark)
        print("Environment: Databricks Notebook (recreated session)")
    except:
        from databricks.connect import DatabricksSession
        from databricks.sdk import WorkspaceClient
        spark = DatabricksSession.builder.getOrCreate()
        dbutils = WorkspaceClient().dbutils
        print("Environment: VS Code")

# --- 2. PARAMETER INITIALIZATION ---
dbutils.widgets.text("catalog", "energy_endava_193")
dbutils.widgets.text("schema", "default") 

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")

print(f"Using catalog: {catalog}")
print(f"Using schema: {schema}")

spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema}")
spark.sql(f"USE SCHEMA {schema}")

print(f"✅ Ingestion Target: {catalog}.{schema}")

# --- 3. DUDETAIL CONFIGURATION ---
TABLE_NAME = "DUDETAIL"
TARGET_TABLE = "nsw_dictionary_second"
CACHE_DIR = "/tmp/nemosis_cache"

if not os.path.exists(CACHE_DIR):
    os.makedirs(CACHE_DIR)

# Drop existing table to reingest fresh data
spark.sql(f"DROP TABLE IF EXISTS {TARGET_TABLE}")
print(f"🗑️ Dropped {TARGET_TABLE} for fresh ingestion")

print(f"\n⚠️  DUDETAIL: Downloading ALL available historical data (no time filtering)\n")

# --- 4. DOWNLOAD ALL DUDETAIL DATA ---
# DUDETAIL is a reference table with time-versioned records (EFFECTIVEDATE + VERSIONNO)
# Download all available data from AEMO archives (starts from ~2015 based on earlier test)

# Use a wide date range to capture all available data
start_date = "2009/01/01 00:00:00"  # Start from earliest possible
end_date = datetime.now().strftime("%Y/%m/%d %H:%M:%S")

print(f"--- ⏳ Downloading {TABLE_NAME}: {start_date} to {end_date} ---")
print(f"ℹ️  This may take several minutes as DUDETAIL contains historical data from ~2015 onwards\n")

try:
    # Download from AEMO
    df_raw = dynamic_data_compiler(
        start_time=start_date,
        end_time=end_date,
        table_name=TABLE_NAME,
        raw_data_location=CACHE_DIR
    )
    
    print(f"ℹ️  Downloaded: {len(df_raw):,} rows (all regions, all time)")
    print(f"ℹ️  No filtering applied - keeping all historical records")
    
    # ⚠️ PYARROW FIX: Dictionary Records Approach
    # Convert to dict records and recreate DataFrame to avoid PyArrow ChunkedArray issues
    # This approach was verified successful in DUDETAILSUMMARY ingestion
    print(f"ℹ️  Applying PyArrow fix: converting to dict records...")
    records = df_raw.to_dict('records')
    df_clean = pd.DataFrame(records)
    df_clean = df_clean.reset_index(drop=True)
    print(f"ℹ️  Recreated DataFrame from {len(records):,} records")
    
    # Create Spark DataFrame
    spark_df = spark.createDataFrame(df_clean)
    
    # Cast numeric columns to DoubleType for consistency
    # DUDETAIL numeric fields: VERSIONNO, REGISTEREDCAPACITY, MAXCAPACITY
    from pyspark.sql.types import IntegralType, FractionalType
    for field in spark_df.schema.fields:
        if isinstance(field.dataType, (IntegralType, FractionalType)):
            spark_df = spark_df.withColumn(field.name, col(field.name).cast(DoubleType()))
    
    # Write to Delta Lake
    spark_df.write.format("delta").mode("overwrite").saveAsTable(TARGET_TABLE)
    
    print(f"✅ Successfully saved {TABLE_NAME} data to {TARGET_TABLE}")
    print(f"📊 Final row count: {spark_df.count():,}")
    
except Exception as e:
    print(f"❌ Error ingesting {TABLE_NAME}: {e}")
    import traceback
    traceback.print_exc()

print(f"\n🚀 {TABLE_NAME} ingestion complete at {catalog}.{schema}.{TARGET_TABLE}")

In [0]:
import os
from nemosis import dynamic_data_compiler
import pandas as pd
from datetime import datetime, timedelta
from pyspark.sql.functions import col
from pyspark.sql.types import DoubleType

# --- 1. ENVIRONMENT & AUTHENTICATION ---
try:
    spark.sql("SELECT 1").collect()
    dbutils
    print("Environment: Databricks Notebook")
except:
    try:
        from pyspark.sql import SparkSession
        from pyspark.dbutils import DBUtils
        spark = SparkSession.builder.getOrCreate()
        dbutils = DBUtils(spark)
        print("Environment: Databricks Notebook (recreated session)")
    except:
        from databricks.connect import DatabricksSession
        from databricks.sdk import WorkspaceClient
        spark = DatabricksSession.builder.getOrCreate()
        dbutils = WorkspaceClient().dbutils
        print("Environment: VS Code")

# --- 2. PARAMETER INITIALIZATION ---
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")

print(f"Using catalog: {catalog}")
print(f"Using schema: {schema}")

spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"USE SCHEMA {schema}")

print(f"✅ Ingestion Target: {catalog}.{schema}")

# --- 3. DISPATCH_UNIT_SCADA CONFIGURATION ---
TABLE_NAME = "DISPATCH_UNIT_SCADA"
TARGET_TABLE = "nsw_scada"
CACHE_DIR = "/tmp/nemosis_cache"

if not os.path.exists(CACHE_DIR):
    os.makedirs(CACHE_DIR)

current_year = datetime.now().year
years_to_process = range(2021, current_year + 1)

# Drop existing table to reingest fresh data
spark.sql(f"DROP TABLE IF EXISTS {TARGET_TABLE}")
print(f"🗑️ Dropped {TARGET_TABLE} for fresh ingestion")

print(f"\n⚠️  DISPATCH_UNIT_SCADA: Using MONTHLY batch processing to avoid memory exhaustion\n")

# --- 4. MONTHLY BATCH PROCESSING LOOP ---
INTEGER_COLUMNS = {'DISPATCHINTERVAL', 'INTERVENTION'}

for year in years_to_process:
    # Determine which months to process for this year
    if year == 2021:
        # Start from October for 5MS transition
        months = range(10, 13)  # Oct, Nov, Dec
    elif year == current_year:
        # Only process completed months + current month if we're past the 1st
        current_month = datetime.now().month
        months = range(1, current_month + 1)
    else:
        # Full year
        months = range(1, 13)
    
    for month in months:
        # Calculate start and end dates for this month
        start_date = datetime(year, month, 1)
        
        # End date is last day of month at 23:55
        if month == 12:
            end_date = datetime(year, 12, 31, 23, 55)
        else:
            # Last day of month = day before next month starts
            next_month = datetime(year, month + 1, 1)
            end_date = next_month - timedelta(minutes=5)
        
        # For current month, end at current time
        if year == current_year and month == datetime.now().month:
            end_date = datetime.now()
        
        print(f"--- ⏳ Downloading {TABLE_NAME}: {start_date.strftime('%Y-%m-%d')} to {end_date.strftime('%Y-%m-%d %H:%M')} ---")
        
        try:
            # Download from AEMO
            df_raw = dynamic_data_compiler(
                start_time=start_date.strftime("%Y/%m/%d %H:%M:%S"),
                end_time=end_date.strftime("%Y/%m/%d %H:%M:%S"),
                table_name=TABLE_NAME,
                raw_data_location=CACHE_DIR
            )
            
            print(f"ℹ️  Downloaded: {len(df_raw):,} rows (all regions)")
            
            # ⚠️ PYARROW FIX: Dictionary Records Approach
            # Convert to dict records and recreate DataFrame to avoid PyArrow ChunkedArray issues
            # This approach was verified successful in DUDETAILSUMMARY ingestion
            print(f"ℹ️  Applying PyArrow fix: converting to dict records...")
            records = df_raw.to_dict('records')
            df_clean = pd.DataFrame(records)
            df_clean = df_clean.reset_index(drop=True)
            print(f"ℹ️  Recreated DataFrame from {len(records):,} records")
            
            # Create Spark DataFrame
            spark_df = spark.createDataFrame(df_clean)
            
            # Cast all numeric columns (except integer identifiers) to DoubleType for consistency
            from pyspark.sql.types import IntegralType, FractionalType
            for field in spark_df.schema.fields:
                if field.name not in INTEGER_COLUMNS:
                    if isinstance(field.dataType, (IntegralType, FractionalType)):
                        spark_df = spark_df.withColumn(field.name, col(field.name).cast(DoubleType()))
            
            # Write to Delta Lake
            # First write (2021-10) uses overwrite, all subsequent writes use append
            if year == 2021 and month == 10:
                spark_df.write.format("delta").mode("overwrite").saveAsTable(TARGET_TABLE)
                print(f"✅ Initialized {TARGET_TABLE} with {year}-{month:02d} data")
            else:
                spark_df.write.format("delta").mode("append") \
                    .option("mergeSchema", "true") \
                    .saveAsTable(TARGET_TABLE)
                print(f"✅ Appended {year}-{month:02d} data to {TARGET_TABLE}")
            
        except Exception as e:
            print(f"❌ Error for {year}-{month:02d}: {e}")
            import traceback
            traceback.print_exc()

print(f"\n🚀 {TABLE_NAME} ingestion complete at {catalog}.{schema}.{TARGET_TABLE}")